# Answer-aligned top-100 reranking (Colab, resumable)

Run cells in order. This is a **new candidate method**, not a required baseline rerun. It tests whether a larger BM25 candidate pool and train-only gold-answer-to-passage alignment can improve a compact cross-encoder reranker. Gold answers and URLs are used **only to create train labels**; dev inference sees the claim and its BM25 candidates. The dev retrieval metric remains the same URL-proxy hit@3 over the same 386 judged claims, and verdict metrics remain on the same 500 dev claims.

The stages are deliberately gated: (1) dev BM25 top 100; (2) train BM25 top 100 only if candidate coverage improves; (3) alignment audit; (4) dev-only cross-encoder pilot; (5) only if useful, out-of-fold train rankings and matched verifier. Every long stage saves checkpoints to Drive. Rerun a cell after interruption. Start with a fresh Colab runtime if you previously imported project scripts, and do not concurrently run this notebook and the older top-20 cross-encoder notebook against the same GPU.

Important: this **fine-tunes a passage reranker**, not the first-stage BGE embedding model. It cannot recover evidence absent from BM25 top 100. An answer-aligned training passage is an automatically generated *silver* label, not verified passage-level relevance.


In [ ]:
import os, subprocess, sys
from pathlib import Path
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
from google.colab import drive
drive.mount('/content/drive')
REPO_URL = 'https://github.com/muyishen2040/10718-ML-In-Practice.git'
REPO_DIR = Path('/content/10718-ML-In-Practice-top100')
if REPO_DIR.exists():
    dirty = subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True).strip()
    if dirty:
        raise RuntimeError('Colab checkout has local edits; save them before pulling.')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
print('Repo revision:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())


In [ ]:
import json, runpy, time
DATA_ROOT = Path('/content/drive/MyDrive/10718_averitec/v2_source_documents')
PROCESSED = DATA_ROOT / 'processed/averitec'
TRAIN_CLAIMS = PROCESSED / 'train.jsonl'
DEV_CORPUS = PROCESSED / 'dev_evidence_corpus.jsonl'
DEV_QRELS = PROCESSED / 'dev_qrels.jsonl'
OLD_DEV_BM25 = DATA_ROOT / 'outputs/bm25/dev_bm25_top20_v2/dev_rankings.jsonl'
ROOT = DATA_ROOT / 'outputs/initial_ml_solution_answer_aligned_top100_v1'
DEV100_DIR = ROOT / 'bm25/dev_bm25_top100_v2'
TRAIN100_DIR = ROOT / 'bm25/train_bm25_top100_v2'
DEV100 = DEV100_DIR / 'dev_rankings.jsonl'
TRAIN100 = TRAIN100_DIR / 'train_rankings.jsonl'
ALIGN_DIR = ROOT / 'alignment/gold_answer_top100_v1'
LABELS = ALIGN_DIR / 'train_alignment.jsonl'
CROSS_DIR = ROOT / 'cross_encoder/minilm_answer_aligned_top100_v1'
DIVERSE_TRAIN = ROOT / 'source_diverse/train_cross_oof_v1'
DIVERSE_DEV = ROOT / 'source_diverse/dev_cross_v1'
VERIFIER = ROOT / 'verifier/cross_diverse_distilbert_v1'
for path in (TRAIN_CLAIMS, DEV_CORPUS, DEV_QRELS, OLD_DEV_BM25):
    if not path.is_file():
        raise FileNotFoundError(path)
for name in ('run_bm25.py', 'run_averitec_archive_bm25.py', 'prepare_answer_aligned_labels.py', 'train_cross_encoder_weak.py', 'run_source_diversity.py', 'train_transformer_verifier.py'):
    if not (REPO_DIR / 'scripts' / name).is_file():
        raise FileNotFoundError(f'New script missing: {name}. Update/push the repo, then rerun setup.')
ROOT.mkdir(parents=True, exist_ok=True)
def run_script(label, filename, arguments):
    script = REPO_DIR / 'scripts' / filename
    old_argv = sys.argv[:]
    sys.argv = [str(script), *map(str, arguments)]
    print(f'\n=== {label} ===\n{script}\nArguments: {arguments}', flush=True)
    started = time.monotonic()
    try:
        runpy.run_path(str(script), run_name='__main__')
    finally:
        sys.argv = old_argv
        print(f'{label}: {(time.monotonic()-started)/60:.1f} minutes this invocation', flush=True)
print('Output root:', ROOT)


## 1. Cheap feasibility check: BM25 top 100 on dev

This reruns lexical retrieval over the existing dev corpus with the same chunking. It does not touch the official test set. Expect this to take around an hour, with a claim-level progress bar and Drive checkpoint.


In [ ]:
if not DEV100.is_file():
    run_script('BM25 dev top 100', 'run_bm25.py', [
        '--split', 'dev', '--data-root', DATA_ROOT, '--corpus', DEV_CORPUS, '--qrels', DEV_QRELS,
        '--output-dir', ROOT / 'bm25', '--run-name', 'dev_bm25_top100_v2',
        '--ranking-k', 100, '--metric-k', 3, '--checkpoint-every', 10, '--resume', '--repair-checkpoint',
    ])
else:
    print('Reusing finalized dev top-100 rankings:', DEV100)
assert DEV100.is_file()


In [ ]:
from scripts.run_bm25 import load_qrels
from src.retrieval.metrics import evaluate_rankings
from src.utils.io import read_jsonl
old_rows = read_jsonl(OLD_DEV_BM25)
new_rows = read_jsonl(DEV100)
old_by_id = {str(row['claim_id']): row for row in old_rows}
new_by_id = {str(row['claim_id']): row for row in new_rows}
assert len(old_by_id) == len(new_by_id) == 500 and set(old_by_id) == set(new_by_id)
mismatch = [cid for cid in old_by_id if [str(x['passage_id']) for x in old_by_id[cid]['retrieved'][:20]] != [str(x['passage_id']) for x in new_by_id[cid]['retrieved'][:20]]]
if mismatch:
    raise RuntimeError(f'Top-100 BM25 is not comparable to old top-20 BM25; {len(mismatch)} claims differ, e.g. {mismatch[:5]}')
qrels = load_qrels(DEV_QRELS)
def retrieval_metrics(rows, k):
    ranking = {str(row['claim_id']): [str(x['passage_id']) for x in row['retrieved']] for row in rows}
    return evaluate_rankings(ranking, qrels, k=k)
old3 = retrieval_metrics(old_rows, 3)
new20 = retrieval_metrics(new_rows, 20)
new100 = retrieval_metrics(new_rows, 100)
gain = new100['evidence_hit_at_100'] - new20['evidence_hit_at_20']
print('Judged dev claims:', new100['judged_claim_count'])
print(f"BM25 hit@3={old3['evidence_hit_at_3']:.4f}; hit@20={new20['evidence_hit_at_20']:.4f}; hit@100={new100['evidence_hit_at_100']:.4f}")
print(f'Additional URL-proxy hit opportunity from positions 21-100: {gain:.4f} ({round(gain*new100["judged_claim_count"])} claims)')
FORCE_CONTINUE = False  # Set True only if you deliberately want to continue despite a small coverage gain.
RUN_LARGER_POOL = gain >= 0.05 or FORCE_CONTINUE
print('Continue to expensive train top-100 retrieval?', RUN_LARGER_POOL)
if not RUN_LARGER_POOL:
    print('Stop here. Keep the earlier BM25 top-20 baselines; a larger rerank pool has little measured URL-proxy upside.')


## 2. Create matched train top-100 rankings and passage labels

Only run these cells when the gate says `True`. The train archive stage can take hours. It processes one official shard at a time, preserves claim checkpoints on Drive, and can be rerun after a disconnect. Do **not** mix these rankings with the previous top-20 train run.


In [ ]:
if RUN_LARGER_POOL and not TRAIN100.is_file():
    for shard in ('train_0_999.zip', 'train_1000_1999.zip', 'train_2000_3067.zip'):
        archive = DATA_ROOT / 'raw/averitec' / shard
        if not archive.is_file():
            raise FileNotFoundError(archive)
        run_script(f'BM25 train top 100 / {shard}', 'run_averitec_archive_bm25.py', [
            '--archive', archive, '--data-root', DATA_ROOT, '--output-dir', ROOT / 'bm25',
            '--run-name', 'train_bm25_top100_v2', '--ranking-k', 100, '--max-words', 160,
            '--overlap-words', 40, '--checkpoint-every', 10, '--resume', '--repair-checkpoint',
        ])
    run_script('Finalize BM25 train top 100', 'run_averitec_archive_bm25.py', [
        '--data-root', DATA_ROOT, '--output-dir', ROOT / 'bm25', '--run-name', 'train_bm25_top100_v2',
        '--ranking-k', 100, '--max-words', 160, '--overlap-words', 40,
        '--checkpoint-every', 10, '--resume', '--repair-checkpoint', '--finalize',
    ])
elif TRAIN100.is_file():
    print('Reusing finalized train top-100 rankings:', TRAIN100)
if RUN_LARGER_POOL:
    assert TRAIN100.is_file()


In [ ]:
if RUN_LARGER_POOL:
    run_script('Train-only answer-to-passage alignment', 'prepare_answer_aligned_labels.py', [
        '--train-claims', TRAIN_CLAIMS, '--train-rankings', TRAIN100,
        '--output-dir', ROOT / 'alignment', '--run-name', 'gold_answer_top100_v1',
        '--top-n', 100, '--min-score', 0.15, '--min-answer-overlap', 2, '--audit-size', 80,
    ])
    alignment = json.loads((ALIGN_DIR / 'alignment_report.json').read_text())
    print(f"Accepted: {alignment['accepted_claim_count']}/{alignment['claim_count']}; matching source in top 100: {alignment['matched_source_in_top_n_count']}")
    print('Review sample passages against gold answers before trusting the labels:', ALIGN_DIR / 'alignment_audit.csv')
    if alignment['accepted_claim_count'] < 300:
        print('Warning: few answer-aligned training claims; continuing is unlikely to yield a strong reranker.')
    FORCE_FINE_TUNE = False
    RUN_FINE_TUNE = alignment['accepted_claim_count'] >= 300 or FORCE_FINE_TUNE
else:
    RUN_FINE_TUNE = False


## 3. Fine-tune and evaluate the passage reranker first

This train-only passage-level selection is a stronger supervision signal than labeling every chunk from a cited URL as relevant. The run selects the BM25/blend weight on a holdout of train claims, then fits on all eligible train claims and reranks dev top 100. The dev qrels are only used to evaluate, never to train or choose the blend. `--dev-only` defers the much longer out-of-fold train scoring.


In [ ]:
CROSS_ARGS = [
    '--train-claims', TRAIN_CLAIMS, '--train-rankings', TRAIN100, '--dev-rankings', DEV100,
    '--dev-qrels', DEV_QRELS, '--positive-labels', LABELS,
    '--output-dir', ROOT / 'cross_encoder', '--run-name', 'minilm_answer_aligned_top100_v1',
    '--top-n', 100, '--negatives', 2, '--learning-rate', '2e-5', '--epochs', 2,
    '--batch-size', 16, '--inference-batch-size', 32, '--max-seq-length', 256,
    '--oof-folds', 2, '--checkpoint-steps', 250, '--checkpoint-claims', 20, '--device', 'cuda',
]
if RUN_FINE_TUNE:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Choose Runtime > Change runtime type > GPU, restart, and rerun setup.')
    print('GPU:', torch.cuda.get_device_name(0))
    run_script('Answer-aligned cross-encoder / dev pilot', 'train_cross_encoder_weak.py', [*CROSS_ARGS, '--dev-only'])
    cross_report = json.loads((CROSS_DIR / 'dev_metrics.json').read_text())
    bm25_hit3 = old3['evidence_hit_at_3']
    cross_hit3 = cross_report['dev']['evidence_hit_at_3']
    print(f'Cross-encoder URL-proxy hit@3={cross_hit3:.4f}; BM25 reference={bm25_hit3:.4f}')
    print('Train-only selection:', cross_report['selected'])
    FORCE_END_TO_END = False
    RUN_END_TO_END = cross_hit3 > bm25_hit3 or FORCE_END_TO_END
    print('Continue to out-of-fold train rankings + matched verifier?', RUN_END_TO_END)
else:
    RUN_END_TO_END = False


## 4. Optional, longer end-to-end comparison

If the reranker improves retrieval, complete leakage-safe out-of-fold train rankings, apply the same source-diversity policy to train and dev, then train the matched DistilBERT verifier. The classifier run can be interrupted and resumed. A retrieval gain is not automatically a verdict gain, so this stage is required before claiming an end-to-end improvement.


In [ ]:
if RUN_END_TO_END:
    run_script('Answer-aligned cross-encoder / OOF train rankings', 'train_cross_encoder_weak.py', CROSS_ARGS)
    assert (CROSS_DIR / 'train_oof_rankings.jsonl').is_file()
    for split, input_path, run_name in [
        ('train', CROSS_DIR / 'train_oof_rankings.jsonl', 'train_cross_oof_v1'),
        ('dev', CROSS_DIR / 'dev_rankings.jsonl', 'dev_cross_v1'),
    ]:
        arguments = ['--input-rankings', input_path, '--output-dir', ROOT / 'source_diverse',
                     '--run-name', run_name, '--split', split, '--max-candidates', 100, '--metric-k', 3]
        if split == 'dev':
            arguments += ['--qrels', DEV_QRELS]
        run_script(f'Source diversity / {split}', 'run_source_diversity.py', arguments)
    assert (DIVERSE_TRAIN / 'train_rankings.jsonl').is_file() and (DIVERSE_DEV / 'dev_rankings.jsonl').is_file()


In [ ]:
if RUN_END_TO_END:
    run_script('Matched DistilBERT verifier', 'train_transformer_verifier.py', [
        '--data-root', DATA_ROOT, '--train-rankings', DIVERSE_TRAIN / 'train_rankings.jsonl',
        '--eval-rankings', DIVERSE_DEV / 'dev_rankings.jsonl',
        '--output-dir', ROOT / 'verifier', '--run-name', 'cross_diverse_distilbert_v1',
        '--model', 'distilbert-base-uncased', '--max-length', 512,
        '--claim-token-budget', 96, '--evidence-token-budget', 110,
        '--learning-rates', '2e-5,5e-5', '--epoch-grid', '2,3',
        '--batch-size', 8, '--gradient-accumulation-steps', 2,
        '--checkpoint-steps', 100, '--seed', 718, '--resume',
    ])
    assert (VERIFIER / 'final_dev_metrics.json').is_file()


In [ ]:
print('Retrieval on the same dev qrels (URL proxy; 386 judged claims):')
for name, path in [('BM25 top 20', OLD_DEV_BM25), ('BM25 top 100', DEV100),
                   ('Answer-aligned cross-encoder', CROSS_DIR / 'dev_rankings.jsonl'),
                   ('Cross-encoder + source diversity', DIVERSE_DEV / 'dev_rankings.jsonl')]:
    if path.is_file():
        metrics = retrieval_metrics(read_jsonl(path), 3)
        print(f"{name:38s} hit@3={metrics['evidence_hit_at_3']:.4f} MRR={metrics['mrr']:.4f}")
print('Verdicts on the same 500 dev claims:')
previous = DATA_ROOT / 'outputs/initial_ml_solution_rerun_bs16/transformer/bm25_top3/final_dev_metrics.json'
for name, path in [('Prior BM25 + DistilBERT', previous), ('New matched method', VERIFIER / 'final_dev_metrics.json')]:
    if path.is_file():
        metrics = json.loads(path.read_text())
        errors = metrics['decision_error_rates']
        print(f"{name:38s} macro-F1={metrics['macro_f1']:.4f} accuracy={metrics['accuracy']:.4f} FR={errors['false_reassurance_rate']:.4f} FA={errors['false_alarm_rate']:.4f}")
print('Interpretation: top-100 hit is a candidate-pool ceiling, not achieved top-three retrieval. Compare verdicts only when both train and dev evidence pipelines match.')
